# Assignment Strategy Comparison

Compares two inventory placement strategies on a **1 500-aisle warehouse** (25 replicas × 60
aisle types covering every pallet-size / category / handling combination).

| | Strategy | Description |
|---|---|---|
| **A** | Uniform | `_uniform_assignment` — random bin selection |
| **B** | Load-Minimizing | `build_load_minimizing_assignment_fn` — greedy L₂ minimisation of predicted L_a |

Both simulations process the **same 1 000 pre-generated batches** drawn with lift-weighted
affinity sampling so batch composition reflects co-ordering correlations within each
`(handling, category)` group.

| Parameter | Value |
|-----------|-------|
| Aisle config types | 60 (48 pallet + 12 singleton) |
| Pallet sizes covered | small, medium, large, extra_large |
| Total aisles | 1 500 (25 replicas × 60 types) |
| Bin fill target | 85% (~114 750 SKUs) |
| Pickers | 25 |
| Batches | 1 000 |
| Affinity | Sparse (≤ 500 SKUs/group, ~3 M entries) — batch sampling, placement, and lift_sum |

Recovered `LoadParams` are loaded from `recovered_params.json` if present, otherwise
defaults (λ=1, γ=1.5) are used.

In [ ]:
import os
import random
import sys
import time as _time
import json as _json
from collections import defaultdict as _dd
from datetime import datetime

_here = os.getcwd()
sys.path.insert(0, os.path.normpath(os.path.join(_here, '..', 'Warehouse')))
sys.path.insert(0, _here)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

from Aisle_Storage import Aisle
from Carton import Carton
from Inventory_Builder import Inventory_Builder, InventoryConfig
from Inventory_Management import Inventory_Manager
from Pick import PickConfig, PickSimulation
from Warehouse_Builder import AisleConfig, Warehouse_Builder, WarehouseConfig
from Workload_Builder import Batch, BatchConfig, Task

from Picking_Analytics import LoadParams, build_load_minimizing_assignment_fn
from Picking_Analytics import sum_lift as _sum_lift
from Picking_Data import (
    BatchStats, TaskStats,
    create_run, init_run_db,
    save_batch_stats, load_batch_stats,
    save_task_stats,  load_task_stats,
)
from Simulation_Analytics import (
    extract_batch_stats, extract_task_stats,
    flag_batch_outliers, flag_task_outliers,
)
from Workload import WorkloadParams

print('Imports OK')

## Configuration

In [ ]:
SEED_WORLD   = 42
SEED_BATCHES = 1337
N_BATCHES    = 1_000
K_PICKERS    = 25

_CATEGORIES  = ['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']
_ALL_SIZES   = ['small', 'medium', 'large', 'extra_large']

# Singleton bins: mixed-size distribution (unchanged from previous design)
_CONV_SIZES  = ['small', 'medium', 'large']
_CONV_PROBS  = [0.25, 0.50, 0.25]
_NCONV_SIZES = ['medium', 'large', 'extra_large']
_NCONV_PROBS = [0.20, 0.50, 0.30]

# ── Aisle config types ────────────────────────────────────────────────────────
# Pallet aisles: one dedicated type per (handling × category × size)  → 48 types
# Singleton aisles: mixed-size distribution per (handling × category) → 12 types
_AISLE_CFGS = []
for _size in _ALL_SIZES:
    for _cat in _CATEGORIES:
        _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'pallet', 10, 10, [_size], None))
        _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'pallet',  8, 10, [_size], None))
for _cat in _CATEGORIES:
    _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'singleton', 10, 10, _CONV_SIZES,  _CONV_PROBS))
    _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'singleton',  8, 10, _NCONV_SIZES, _NCONV_PROBS))

_N_TYPES      = len(_AISLE_CFGS)      # 60
_REPLICAS     = 25
_TOTAL_AISLES = _N_TYPES * _REPLICAS   # 1 500

WAREHOUSE_CFG = WarehouseConfig(
    total_aisles  = _TOTAL_AISLES,
    aisle_splits  = [1 / _N_TYPES] * _N_TYPES,
    aisle_configs = _AISLE_CFGS,
)

PICK_CFG = PickConfig(
    num_pickers      = K_PICKERS,
    x_move_time      = 1.0,
    y_move_time      = 0.5,
    pick_intercept   = 1.0,
    pick_weight_coef = 0.02,
    pick_volume_coef = 1e-4,
    cart_swap_coef   = 5.0,
)
WP = WorkloadParams.from_pick_config(PICK_CFG)

_ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
DB_PATH = os.path.join(_here, f'comparison_sim_{_ts}.db')

_A_COL = '#5b9bd5'   # blue  — Uniform
_B_COL = '#f4a030'   # amber — Load-Minimizing

print(f'Aisle config types : {_N_TYPES}  (48 pallet + 12 singleton)')
print(f'Total aisles       : {_TOTAL_AISLES}  ({_REPLICAS} replicas × {_N_TYPES} types)')
print(f'Pickers            : {K_PICKERS}')
print(f'Batches            : {N_BATCHES:,}')
print(f'DB path            : {DB_PATH}')

## Build Inventory & Affinity

A probe build measures the exact bin count, then inventory is sized to 85% fill.

A **sparse affinity** (≤ 500 SKUs per `(handling, category)` group, ~3 M entries) is built
from the inventory before any warehouse is stocked.  It is used for:
- **Batch sampling** — lift-weighted selection draws co-ordering correlated SKUs; the
  partner-map is cached on first call so the O(|affinity|) build is paid only once
- `build_load_minimizing_assignment_fn` — co-locating high-affinity SKUs in warehouse B
- `extract_task_stats` — computing `lift_sum` on each task

In [ ]:
_FILL_TARGET     = 0.85
_BATCH_MEAN_FRAC = 0.10
_BATCH_STD_FRAC  = 0.03
_MAX_PER_GROUP   = 500   # SKUs per (handling, category) group in sparse affinity

# ── Probe to measure exact bin count ─────────────────────────────────────────
Carton.next_sku     = 1
Aisle.next_aisle_id = 1
random.seed(SEED_WORLD)
_probe     = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()
TOTAL_BINS = len(_probe.bins)
N_SKUS     = int(_FILL_TARGET * TOTAL_BINS)
del _probe

INV_CFG = InventoryConfig(
    num_skus           = N_SKUS,
    handling_splits    = [0.5, 0.5],
    category_splits    = [1 / 6] * 6,
    singleton_fraction = 0.5,
)

Carton.next_sku = 1
random.seed(SEED_WORLD + 10)
inventory = Inventory_Builder().from_config(INV_CFG).build()

# ── Sparse affinity from inventory (pre-placement) ────────────────────────────
# Groups by (handling, category) — only co-located SKUs can have correlated demand.
# Capped at _MAX_PER_GROUP per group; 12 groups × C(500,2) × 2 ≈ 3 M entries.
print('Building sparse affinity...', end=' ', flush=True)
t0 = _time.perf_counter()
AFFINITY = inventory.affinity_matrix(max_per_group=_MAX_PER_GROUP)
print(f'{len(AFFINITY):,} entries  ({_time.perf_counter() - t0:.2f}s)')

BATCH_CFG = BatchConfig(
    inventory_size = N_SKUS,
    mean_fraction  = _BATCH_MEAN_FRAC,
    std_fraction   = _BATCH_STD_FRAC,
)

# ── Load recovered LoadParams or use defaults ─────────────────────────────────
_param_path = os.path.join(_here, 'recovered_params.json')
if os.path.exists(_param_path):
    _p = _json.load(open(_param_path))
    LOAD_PARAMS = LoadParams(lambda_=_p['lambda_'], k=1.0, gamma=_p['gamma'])
    print(f'Recovered params  λ={LOAD_PARAMS.lambda_:.4f}  γ={LOAD_PARAMS.gamma:.4f}')
else:
    LOAD_PARAMS = LoadParams(lambda_=1.0, k=1.0, gamma=1.5)
    print('recovered_params.json not found — using defaults (λ=1.0  γ=1.5)')

print(f'\nWarehouse bins  : {TOTAL_BINS:,}')
print(f'SKU catalog     : {N_SKUS:,}  ({_FILL_TARGET:.0%} fill target)')
print(f'Avg batch size  : ~{int(N_SKUS * _BATCH_MEAN_FRAC):,} SKUs')

## Build Warehouses A and B

In [ ]:
# ── Warehouse A: uniform (random) assignment ───────────────────────────────────
Aisle.next_aisle_id = 1
random.seed(SEED_WORLD)           # same seed → identical bin-size layout as probe
warehouse_A = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()

random.seed(SEED_WORLD + 100)
manager_A   = Inventory_Manager(warehouse_A)
manager_A.enqueue_all(inventory.cartons, quantity=1)

placed_A = len(manager_A.unavailable)
print(f'Warehouse A (uniform)   : {placed_A:,} / {TOTAL_BINS:,} bins filled  ({placed_A / TOTAL_BINS:.1%})')

# ── Warehouse B: load-minimising assignment ────────────────────────────────────
Aisle.next_aisle_id = 1
random.seed(SEED_WORLD)           # identical bin-size layout
warehouse_B = Warehouse_Builder().from_config(WAREHOUSE_CFG).build()

print('Stocking warehouse B (load-minimizing)...', end=' ', flush=True)
t0        = _time.perf_counter()
random.seed(SEED_WORLD + 200)
assign_fn = build_load_minimizing_assignment_fn(LOAD_PARAMS, AFFINITY, WP)
manager_B = Inventory_Manager(warehouse_B, assignment_fn=assign_fn)
manager_B.enqueue_all(inventory.cartons, quantity=1)
print(f'{_time.perf_counter() - t0:.1f}s')

placed_B = len(manager_B.unavailable)
print(f'Warehouse B (load-min)  : {placed_B:,} / {TOTAL_BINS:,} bins filled  ({placed_B / TOTAL_BINS:.1%})')

# ── Aisle metadata maps (same layout for A and B) ─────────────────────────────
AISLE_SIZE_MAP     = {a.aisle_id: a.storage_size  for a in warehouse_A.aisles}
AISLE_UNITTYPE_MAP = {a.aisle_id: a.unit_type     for a in warehouse_A.aisles}
AISLE_HANDLING_MAP = {a.aisle_id: a.handling_type for a in warehouse_A.aisles}

## Lift-Sum Distribution after Assignment

With 1 500 aisles a bar chart is illegible; instead we compare the distribution of per-aisle
lift sums as overlaid histograms and report summary statistics.

In [ ]:
def _aisle_lift_sums(wh):
    by_aisle = _dd(list)
    for b in wh.bins:
        if b.storage is not None:
            by_aisle[b.location[0]].append(b.storage.carton.sku)
    return {aid: _sum_lift(skus, AFFINITY) for aid, skus in by_aisle.items()}

lifts_A   = _aisle_lift_sums(warehouse_A)
lifts_B   = _aisle_lift_sums(warehouse_B)
aisle_ids = sorted(set(lifts_A) | set(lifts_B))
la = np.array([lifts_A.get(a, 0.0) for a in aisle_ids])
lb = np.array([lifts_B.get(a, 0.0) for a in aisle_ids])

print(f'Σ(lift²) Uniform   : {(la**2).sum():>14,.0f}   mean/aisle={la.mean():.2f}  std={la.std():.2f}')
print(f'Σ(lift²) Load-min  : {(lb**2).sum():>14,.0f}   mean/aisle={lb.mean():.2f}  std={lb.std():.2f}')
print(f'L2 ratio B/A = {(lb**2).sum() / max((la**2).sum(), 1e-9):.4f}  (< 1 means load-min reduced congestion potential)')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Per-Aisle Lift Sum after Assignment  (static snapshot, before any picks)',
             fontsize=12, fontweight='bold')

# Overlaid histograms — readable at 1 500 aisles
for ax, (vals_a, vals_b) in [(
    axes[0],
    (la[la > 0], lb[lb > 0]),
)]:
    ax.hist(vals_a, bins=50, color=_A_COL, alpha=0.55, edgecolor='white',
            label=f'Uniform (A)  μ={vals_a.mean():.1f}')
    ax.hist(vals_b, bins=50, color=_B_COL, alpha=0.55, edgecolor='white',
            label=f'Load-Min (B)  μ={vals_b.mean():.1f}')
    ax.axvline(vals_a.mean(), color=_A_COL, lw=2, linestyle='--')
    ax.axvline(vals_b.mean(), color=_B_COL, lw=2, linestyle='--')
    ax.set_title('Lift sum distribution  (aisles with lift > 0)', fontsize=10)
    ax.set_xlabel('Σ lift (ordered pairs in aisle)')
    ax.set_ylabel('Aisle count')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

# CDF comparison
for vals, label, color in [(np.sort(la), 'Uniform (A)', _A_COL),
                            (np.sort(lb), 'Load-Min (B)', _B_COL)]:
    cdf = np.arange(1, len(vals) + 1) / len(vals)
    axes[1].plot(vals, cdf, color=color, lw=2, label=label)
axes[1].set_xlabel('Per-aisle lift sum')
axes[1].set_ylabel('Cumulative fraction of aisles')
axes[1].set_title('CDF of per-aisle lift sum', fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Initialise Database & Pre-generate Batches

In [ ]:
init_run_db(DB_PATH)
RUN_A = create_run(DB_PATH, 'uniform_assignment')
RUN_B = create_run(DB_PATH, 'load_minimizing_assignment')
print(f'run_id A (uniform)   : {RUN_A}')
print(f'run_id B (load-min)  : {RUN_B}')

# ── Pre-generate all batches ──────────────────────────────────────────────────
# affinity=AFFINITY enables lift-weighted sampling via the numpy-accelerated
# _lift_weighted_sample; partner_map is cached after the first call.
# Identical batch sequence for both warehouses; differences come from layout only.
random.seed(SEED_BATCHES)
print(f'\nGenerating {N_BATCHES:,} batches (affinity=AFFINITY)...', end=' ', flush=True)
t0      = _time.perf_counter()
BATCHES = [Batch(BATCH_CFG, inventory, affinity=AFFINITY) for _ in range(N_BATCHES)]
elapsed = _time.perf_counter() - t0
sizes   = [len(b.items) for b in BATCHES]
print(f'{elapsed:.1f}s')
print(f'Batch sizes  mean={np.mean(sizes):.1f}  std={np.std(sizes):.1f}  '
      f'min={min(sizes)}  max={max(sizes)}')

## Simulation Loop — 1 000 Batch Pairs

In [ ]:
_CHECKPOINT = 100

all_bs_A: list[BatchStats] = [];  all_bs_B: list[BatchStats] = []
all_ts_A: list[TaskStats]  = [];  all_ts_B: list[TaskStats]  = []
_pb_A: list[BatchStats] = [];     _pb_B: list[BatchStats] = []
_pt_A: list[TaskStats]  = [];     _pt_B: list[TaskStats]  = []

t_loop  = _time.perf_counter()
skipped = 0

for _i, _batch in enumerate(BATCHES):
    _ta = Task.from_batch(_batch, warehouse_A)
    _tb = Task.from_batch(_batch, warehouse_B)

    if not _ta or not _tb:
        skipped += 1
        continue

    _ea = PickSimulation(_ta, PICK_CFG).run()
    _eb = PickSimulation(_tb, PICK_CFG).run()

    _bsA = extract_batch_stats(_ea, batch_id=_i, k_pickers=K_PICKERS, run_id=RUN_A)
    _bsB = extract_batch_stats(_eb, batch_id=_i, k_pickers=K_PICKERS, run_id=RUN_B)
    # AFFINITY is the sparse pre-placement dict; provides lift_sum for placed pairs
    _tsA = extract_task_stats(_ea, _ta, batch_id=_i, affinity=AFFINITY, wp=WP, run_id=RUN_A)
    _tsB = extract_task_stats(_eb, _tb, batch_id=_i, affinity=AFFINITY, wp=WP, run_id=RUN_B)

    _pb_A.append(_bsA);  all_bs_A.append(_bsA)
    _pb_B.append(_bsB);  all_bs_B.append(_bsB)
    _pt_A.extend(_tsA);  all_ts_A.extend(_tsA)
    _pt_B.extend(_tsB);  all_ts_B.extend(_tsB)

    if len(_pb_A) >= _CHECKPOINT:
        save_batch_stats(DB_PATH, RUN_A, _pb_A);  save_batch_stats(DB_PATH, RUN_B, _pb_B)
        save_task_stats( DB_PATH, RUN_A, _pt_A);  save_task_stats( DB_PATH, RUN_B, _pt_B)
        _rate = (_i + 1) / (_time.perf_counter() - t_loop)
        print(f'Batch {_i+1:4d}/{N_BATCHES}  '
              f'dur_A={_bsA.duration:.1f}  dur_B={_bsB.duration:.1f}  '
              f'{_rate:.1f} pairs/s', flush=True)
        _pb_A.clear(); _pb_B.clear(); _pt_A.clear(); _pt_B.clear()

if _pb_A:
    save_batch_stats(DB_PATH, RUN_A, _pb_A);  save_batch_stats(DB_PATH, RUN_B, _pb_B)
    save_task_stats( DB_PATH, RUN_A, _pt_A);  save_task_stats( DB_PATH, RUN_B, _pt_B)

print(f'\nDone  {N_BATCHES - skipped} batch-pairs in '
      f'{_time.perf_counter() - t_loop:.1f}s  ({skipped} skipped)')

In [ ]:
bs_raw_A = load_batch_stats(DB_PATH, RUN_A);  bs_raw_B = load_batch_stats(DB_PATH, RUN_B)
ts_raw_A = load_task_stats( DB_PATH, RUN_A);  ts_raw_B = load_task_stats( DB_PATH, RUN_B)

bs_fA = flag_batch_outliers(bs_raw_A);  bs_fB = flag_batch_outliers(bs_raw_B)
ts_fA = flag_task_outliers(ts_raw_A);   ts_fB = flag_task_outliers(ts_raw_B)

bs_cA = [s for s in bs_fA if not s.is_outlier];  bs_cB = [s for s in bs_fB if not s.is_outlier]
ts_cA = [s for s in ts_fA if not s.is_outlier];  ts_cB = [s for s in ts_fB if not s.is_outlier]

def _bdf(stats):
    return pd.DataFrame([{
        'batch_id': s.batch_id, 'duration': s.duration,
        'num_tasks': s.num_tasks, 'total_items': s.total_items,
        'completion_rate': s.total_items / s.duration if s.duration > 0 else 0.0,
        'avg_concurrent_pickers': s.avg_concurrent_pickers,
        'picking_pct': s.picking_pct * 100, 'traveling_pct': s.traveling_pct * 100,
    } for s in stats])

def _tdf(stats):
    return pd.DataFrame([{
        'batch_id'   : s.batch_id,
        'aisle_id'   : s.aisle_id,
        'duration'   : s.duration,
        'W_a'        : s.W_a,
        'lift_sum'   : s.lift_sum,
        'num_bins'   : s.num_bins_visited,
        'total_items': s.total_items,
        'pallet_size': AISLE_SIZE_MAP.get(s.aisle_id),
        'unit_type'  : AISLE_UNITTYPE_MAP.get(s.aisle_id),
        'handling'   : AISLE_HANDLING_MAP.get(s.aisle_id),
    } for s in stats])

df_bA = _bdf(bs_cA);  df_bB = _bdf(bs_cB)
df_tA = _tdf(ts_cA);  df_tB = _tdf(ts_cB)

for lbl, bs_c, ts_c, bs_f in [
    ('A Uniform ', bs_cA, ts_cA, bs_fA),
    ('B Load-Min', bs_cB, ts_cB, bs_fB),
]:
    n_out = sum(s.is_outlier for s in bs_f)
    print(f'{lbl}: {len(bs_c):,} clean batches  ({n_out} outliers)  |  {len(ts_c):,} clean tasks')

## Summary Statistics

In [ ]:
_BCOLS = ['duration', 'completion_rate', 'avg_concurrent_pickers', 'picking_pct', 'traveling_pct']
_TCOLS = ['duration', 'W_a', 'lift_sum', 'num_bins']

summ_b = pd.concat(
    [df_bA[_BCOLS].agg(['mean', 'median', 'std']).T,
     df_bB[_BCOLS].agg(['mean', 'median', 'std']).T],
    axis=1, keys=['Uniform (A)', 'Load-Min (B)'],
).round(3)

summ_t = pd.concat(
    [df_tA[_TCOLS].agg(['mean', 'median', 'std']).T,
     df_tB[_TCOLS].agg(['mean', 'median', 'std']).T],
    axis=1, keys=['Uniform (A)', 'Load-Min (B)'],
).round(3)

print('=== Batch Statistics ===')
display(summ_b)
print('\n=== Task Statistics ===')
display(summ_t)

print('\n=== Key Metric Deltas  (Load-Min vs Uniform) ===')
_METRICS = [
    ('duration',               True,  'lower is better'),
    ('completion_rate',        False, 'higher is better'),
    ('avg_concurrent_pickers', False, 'higher is better'),
    ('picking_pct',            False, 'higher is better'),
]
for col, lower_better, note in _METRICS:
    a_mu = df_bA[col].mean();  b_mu = df_bB[col].mean()
    delta = (b_mu - a_mu) / abs(a_mu) * 100
    good  = (lower_better and delta < 0) or (not lower_better and delta > 0)
    print(f'  {col:35s}: {delta:+.2f}%  ({note})  {"✓" if good else "✗"}')

## Plot 1 — Batch Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Batch Completion Time Distribution', fontsize=13, fontweight='bold')

for ax, data, label, color in [
    (axes[0], df_bA['duration'].values, 'A — Uniform',         _A_COL),
    (axes[1], df_bB['duration'].values, 'B — Load-Minimizing', _B_COL),
]:
    ax.hist(data, bins=40, color=color, alpha=0.65, edgecolor='white')
    if len(data) > 1:
        kde = gaussian_kde(data, bw_method='silverman')
        xs  = np.linspace(data.min(), data.max(), 400)
        ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 40,
                color=color, lw=2)
    ax.axvline(data.mean(),     color='red',    lw=1.5, linestyle='--',
               label=f'Mean   {data.mean():.1f}')
    ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',
               label=f'Median {np.median(data):.1f}')
    ax.set_xlabel('Batch duration  (sim time units)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
    ax.legend(fontsize=8);  ax.grid(axis='y', alpha=0.3)

plt.tight_layout();  plt.show()

## Plot 2 — Task Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Task (Aisle) Completion Time Distribution', fontsize=13, fontweight='bold')

for ax, data, label, color in [
    (axes[0], df_tA['duration'].values, 'A — Uniform',         _A_COL),
    (axes[1], df_tB['duration'].values, 'B — Load-Minimizing', _B_COL),
]:
    ax.hist(data, bins=50, color=color, alpha=0.65, edgecolor='white')
    if len(data) > 1:
        kde = gaussian_kde(data, bw_method='silverman')
        xs  = np.linspace(data.min(), data.max(), 400)
        ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 50,
                color=color, lw=2)
    ax.axvline(data.mean(),     color='red',    lw=1.5, linestyle='--',
               label=f'Mean   {data.mean():.1f}')
    ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',
               label=f'Median {np.median(data):.1f}')
    ax.set_xlabel('Task duration  (sim time units)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
    ax.legend(fontsize=8);  ax.grid(axis='y', alpha=0.3)

plt.tight_layout();  plt.show()

## Plot 3 — Batch Completion Rate Over Time

In [ ]:
_WIN = 50

def _roll(df, col):
    return df.sort_values('batch_id')[col].rolling(_WIN, min_periods=1).mean().values

bid_A = df_bA.sort_values('batch_id')['batch_id'].values
bid_B = df_bB.sort_values('batch_id')['batch_id'].values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
fig.suptitle(f'Batch Completion Rate  (rolling {_WIN}-batch window)',
             fontsize=13, fontweight='bold')

ax1.plot(bid_A, _roll(df_bA, 'completion_rate'), color=_A_COL, lw=2, label='Uniform (A)')
ax1.plot(bid_B, _roll(df_bB, 'completion_rate'), color=_B_COL, lw=2, label='Load-Min (B)')
ax1.set_ylabel('Items / time unit', fontsize=10)
ax1.set_title('Throughput rate', fontsize=10)
ax1.legend(fontsize=9);  ax1.grid(alpha=0.3)

ax2.plot(bid_A, _roll(df_bA, 'duration'), color=_A_COL, lw=2, label='Uniform (A)')
ax2.plot(bid_B, _roll(df_bB, 'duration'), color=_B_COL, lw=2, label='Load-Min (B)')
ax2.set_xlabel('Batch ID', fontsize=10)
ax2.set_ylabel('Duration  (time units)', fontsize=10)
ax2.set_title('Batch completion time', fontsize=10)
ax2.legend(fontsize=9);  ax2.grid(alpha=0.3)

plt.tight_layout();  plt.show()

## Plot 4 — Picker Concurrency Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Picker Concurrency  (avg pickers simultaneously in picking state)',
             fontsize=12, fontweight='bold')

for ax, data, label, color in [
    (axes[0], df_bA['avg_concurrent_pickers'].values, 'A — Uniform',         _A_COL),
    (axes[1], df_bB['avg_concurrent_pickers'].values, 'B — Load-Minimizing', _B_COL),
]:
    ax.hist(data, bins=35, color=color, alpha=0.70, edgecolor='white')
    if len(data) > 1 and data.max() > data.min():
        kde = gaussian_kde(data, bw_method='silverman')
        xs  = np.linspace(data.min(), data.max(), 400)
        ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 35,
                color=color, lw=2)
    ax.axvline(data.mean(),     color='red',    lw=1.5, linestyle='--',
               label=f'Mean   {data.mean():.2f}')
    ax.axvline(np.median(data), color='orange', lw=1.5, linestyle=':',
               label=f'Median {np.median(data):.2f}')
    ax.axvline(K_PICKERS, color='grey', lw=1.0, linestyle='-.', label=f'Max ({K_PICKERS})')
    ax.set_xlabel('Avg concurrent pickers', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{label}  (n={len(data):,})', fontsize=10)
    ax.legend(fontsize=8);  ax.grid(axis='y', alpha=0.3)

plt.tight_layout();  plt.show()

## Plot 5 — Picker Utilisation Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('Picker Utilisation Breakdown', fontsize=13, fontweight='bold')

bp = axes[0].boxplot(
    [df_bA['picking_pct'].values, df_bB['picking_pct'].values,
     df_bA['traveling_pct'].values, df_bB['traveling_pct'].values],
    labels=['Pick A', 'Pick B', 'Travel A', 'Travel B'],
    patch_artist=True, medianprops=dict(color='black', lw=2),
)
for patch, color in zip(bp['boxes'], [_A_COL, _B_COL, _A_COL, _B_COL]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[0].set_title('Picking vs Traveling %', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(df_bA['picking_pct'].values, bins=30, color=_A_COL, alpha=0.55,
             edgecolor='white', label=f'Uniform  μ={df_bA["picking_pct"].mean():.1f}%')
axes[1].hist(df_bB['picking_pct'].values, bins=30, color=_B_COL, alpha=0.55,
             edgecolor='white', label=f'Load-Min μ={df_bB["picking_pct"].mean():.1f}%')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[1].set_xlabel('Picking %', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('Picking % — overlaid', fontsize=10)
axes[1].legend(fontsize=8);  axes[1].grid(axis='y', alpha=0.3)

_x  = np.arange(2)
_pk = [df_bA['picking_pct'].mean(),  df_bB['picking_pct'].mean()]
_tr = [df_bA['traveling_pct'].mean(), df_bB['traveling_pct'].mean()]
axes[2].bar(_x, _pk, width=0.5, color=[_A_COL, _B_COL], alpha=0.85, label='Picking')
axes[2].bar(_x, _tr, width=0.5, bottom=_pk, color='#70ad47', alpha=0.85, label='Traveling')
axes[2].set_xticks(_x);  axes[2].set_xticklabels(['Uniform (A)', 'Load-Min (B)'])
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
axes[2].set_ylabel('Mean fraction (%)', fontsize=10)
axes[2].set_title('Aggregate mean split', fontsize=10)
axes[2].legend(fontsize=8);  axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout();  plt.show()

## Plot 6 — Pallet Size Analysis: A vs B

Because every pallet aisle is dedicated to a single storage size, we can compare how load-minimizing
assignment affects task duration and item throughput broken out by size.

In [ ]:
_SIZE_ORDER  = ['small', 'medium', 'large', 'extra_large']
_SIZE_LABELS = ['Small', 'Medium', 'Large', 'Extra-Large']

def _pallet_df(df):
    d = df[(df['unit_type'] == 'pallet') & (df['pallet_size'].notna())].copy()
    d['pallet_size'] = pd.Categorical(d['pallet_size'], categories=_SIZE_ORDER, ordered=True)
    return d

dp_A = _pallet_df(df_tA)
dp_B = _pallet_df(df_tB)

# ── Mean task duration per size: A vs B ───────────────────────────────────────
mean_dur_A = [dp_A[dp_A['pallet_size'] == s]['duration'].mean() for s in _SIZE_ORDER]
mean_dur_B = [dp_B[dp_B['pallet_size'] == s]['duration'].mean() for s in _SIZE_ORDER]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Pallet-Aisle Task Analysis by Storage Size  —  A vs B  (clean tasks)',
             fontsize=13, fontweight='bold')

# Side-by-side mean task duration
x = np.arange(len(_SIZE_ORDER))
w = 0.38
axes[0].bar(x - w/2, mean_dur_A, width=w, color=_A_COL, alpha=0.85, label='Uniform (A)')
axes[0].bar(x + w/2, mean_dur_B, width=w, color=_B_COL, alpha=0.85, label='Load-Min (B)')
axes[0].set_xticks(x);  axes[0].set_xticklabels(_SIZE_LABELS)
axes[0].set_ylabel('Mean task duration  (time units)', fontsize=10)
axes[0].set_title('Mean task duration per pallet size', fontsize=10)
axes[0].legend(fontsize=9);  axes[0].grid(axis='y', alpha=0.3)

# % improvement (B vs A) per size
delta_pct = [(b - a) / abs(a) * 100 if a else 0
             for a, b in zip(mean_dur_A, mean_dur_B)]
bar_cols = [_B_COL if d < 0 else '#c00000' for d in delta_pct]
axes[1].bar(_SIZE_LABELS, delta_pct, color=bar_cols, alpha=0.85)
axes[1].axhline(0, color='black', lw=1)
for i, d in enumerate(delta_pct):
    axes[1].text(i, d + (0.3 if d >= 0 else -0.5), f'{d:.1f}%',
                 ha='center', fontsize=9)
axes[1].set_ylabel('Δ task duration  (B − A) / A  %', fontsize=10)
axes[1].set_title('Duration improvement per size  (amber = faster with load-min)', fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].grid(axis='y', alpha=0.3)

# Overlaid KDE: task duration per size, showing A (solid) vs B (dashed)
_SIZE_COLORS = ['#9dc3e6', '#5b9bd5', '#2e75b6', '#1f4e79']
for size, color, label in zip(_SIZE_ORDER, _SIZE_COLORS, _SIZE_LABELS):
    vA = dp_A[dp_A['pallet_size'] == size]['duration'].values
    vB = dp_B[dp_B['pallet_size'] == size]['duration'].values
    if len(vA) > 1:
        kA = gaussian_kde(vA, bw_method='silverman')
        xs = np.linspace(min(vA.min(), vB.min()), max(vA.max(), vB.max()), 300)
        axes[2].plot(xs, kA(xs), color=color, lw=2, linestyle='-',
                     label=f'{label} A  μ={vA.mean():.1f}')
    if len(vB) > 1:
        kB = gaussian_kde(vB, bw_method='silverman')
        xs = np.linspace(min(vA.min(), vB.min()), max(vA.max(), vB.max()), 300)
        axes[2].plot(xs, kB(xs), color=color, lw=2, linestyle='--',
                     label=f'{label} B  μ={vB.mean():.1f}')
axes[2].set_xlabel('Task duration  (time units)', fontsize=10)
axes[2].set_ylabel('Density', fontsize=10)
axes[2].set_title('KDE: solid=Uniform, dashed=Load-Min', fontsize=10)
axes[2].legend(fontsize=7, ncol=2)
axes[2].grid(alpha=0.3)

plt.tight_layout();  plt.show()

# ── Handling breakdown: conveyable vs non-conveyable per size ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mean Task Duration per Pallet Size × Handling Type  —  A vs B',
             fontsize=12, fontweight='bold')

for ax, handling in [(axes[0], 'conveyable'), (axes[1], 'non-conveyable')]:
    mA = [dp_A[(dp_A['pallet_size']==s) & (dp_A['handling']==handling)]['duration'].mean()
          for s in _SIZE_ORDER]
    mB = [dp_B[(dp_B['pallet_size']==s) & (dp_B['handling']==handling)]['duration'].mean()
          for s in _SIZE_ORDER]
    ax.bar(x - w/2, mA, width=w, color=_A_COL, alpha=0.85, label='Uniform (A)')
    ax.bar(x + w/2, mB, width=w, color=_B_COL, alpha=0.85, label='Load-Min (B)')
    ax.set_xticks(x);  ax.set_xticklabels(_SIZE_LABELS)
    ax.set_title(f'{handling.capitalize()} pallet aisles', fontsize=10)
    ax.set_ylabel('Mean task duration  (time units)', fontsize=10)
    ax.legend(fontsize=9);  ax.grid(axis='y', alpha=0.3)

plt.tight_layout();  plt.show()

print('\n=== Mean task duration by pallet size: A vs B ===')
_size_summary = pd.DataFrame({
    'Uniform_mean'  : [dp_A[dp_A['pallet_size']==s]['duration'].mean() for s in _SIZE_ORDER],
    'LoadMin_mean'  : [dp_B[dp_B['pallet_size']==s]['duration'].mean() for s in _SIZE_ORDER],
    'delta_pct'     : delta_pct,
    'n_tasks_A'     : [len(dp_A[dp_A['pallet_size']==s]) for s in _SIZE_ORDER],
}, index=_SIZE_LABELS).round(3)
display(_size_summary)

## Per-Aisle Mean Task Duration — A vs B

With 1 500 aisles a side-by-side bar chart is illegible; instead we show a histogram of
per-aisle mean durations (overlaid A vs B) and the distribution of % deltas.

In [ ]:
aisle_A   = df_tA.groupby('aisle_id')['duration'].mean().rename('A')
aisle_B   = df_tB.groupby('aisle_id')['duration'].mean().rename('B')
aisle_cmp = pd.concat([aisle_A, aisle_B], axis=1).dropna()
aisle_cmp['delta_pct'] = (aisle_cmp['B'] - aisle_cmp['A']) / aisle_cmp['A'].abs() * 100

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Per-Aisle Mean Task Duration  —  A vs B  (clean tasks only)',
             fontsize=13, fontweight='bold')

# Overlaid histogram of per-aisle means
axes[0].hist(aisle_cmp['A'], bins=50, color=_A_COL, alpha=0.55,
             edgecolor='white', label=f'Uniform (A)  μ={aisle_cmp["A"].mean():.1f}')
axes[0].hist(aisle_cmp['B'], bins=50, color=_B_COL, alpha=0.55,
             edgecolor='white', label=f'Load-Min (B)  μ={aisle_cmp["B"].mean():.1f}')
axes[0].set_xlabel('Per-aisle mean task duration', fontsize=10)
axes[0].set_ylabel('Aisle count', fontsize=10)
axes[0].set_title('Distribution of aisle mean durations', fontsize=10)
axes[0].legend(fontsize=9);  axes[0].grid(axis='y', alpha=0.3)

# CDF comparison
for vals, label, color in [(np.sort(aisle_cmp['A'].values), 'Uniform (A)', _A_COL),
                            (np.sort(aisle_cmp['B'].values), 'Load-Min (B)', _B_COL)]:
    cdf = np.arange(1, len(vals) + 1) / len(vals)
    axes[1].plot(vals, cdf, color=color, lw=2, label=label)
axes[1].set_xlabel('Per-aisle mean task duration', fontsize=10)
axes[1].set_ylabel('Cumulative fraction of aisles', fontsize=10)
axes[1].set_title('CDF of per-aisle mean duration', fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
axes[1].legend(fontsize=9);  axes[1].grid(alpha=0.3)

# Histogram of % delta
delta_vals = aisle_cmp['delta_pct'].values
axes[2].hist(delta_vals, bins=50, color='#9dc3e6', alpha=0.8, edgecolor='white')
axes[2].axvline(0, color='black', lw=1.5, linestyle='--')
axes[2].axvline(delta_vals.mean(), color='red', lw=1.5, linestyle='--',
                label=f'Mean {delta_vals.mean():.2f}%')
axes[2].set_xlabel('Δ mean duration  (B − A) / A  %', fontsize=10)
axes[2].set_ylabel('Aisle count', fontsize=10)
axes[2].set_title('Per-aisle % duration change  (B vs A)', fontsize=10)
axes[2].xaxis.set_major_formatter(mticker.PercentFormatter())
axes[2].legend(fontsize=9);  axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout();  plt.show()

improved = (aisle_cmp['delta_pct'] < 0).sum()
print(f'Aisles faster with load-min : {improved} / {len(aisle_cmp)}')
print(f'Mean delta across all aisles: {aisle_cmp["delta_pct"].mean():.2f}%')